In [1]:
# workaround for incredibly stupid vscode on snap bug
from os import environ
krb5ccname = environ["KRB5CCNAME"]
if "hostfs" in krb5ccname:
    krb5ccname = krb5ccname.replace("/var/lib/snapd/hostfs", "")
    environ["KRB5CCNAME"] = krb5ccname

In [2]:
from analysis_framework import Dataset
from ReweightingHelper import ReweightingHelper
from AltSetupHandler import AltSetupHandler

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x7f68800
OBJ: TStyle	ildStyle	ILD Style : 0 at: 0x8094240


In [3]:
# CLD
# x_angle = 0.030 # rad
# ILD
x_angle = 0.014 # rad
n_threads = 12
# prod = False
prod = True
no_rvec = True
# write_outputs = False
write_outputs = True
# dataset_path = "data/datasets/selected-objects/test.json"
# output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/oo-sqme/test"
# output_meta_path = "data/datasets/oo-sqme"
# output_meta = f"{output_meta_path}/test.json"
# checked_output_meta = f"{output_meta_path}/checked-test.json"
output_collections = r"(\w*sqme\w*)"
if prod:
    dataset_path = "data/datasets/fitted-objects/signal-only-clean.json"
    output_path = "root://eosuser.cern.ch//eos/user/l/lreichen/TGC2/data/oo-sqme/signal-only-kinfit"
    output_meta_path = "data/datasets/oo-sqme"
    output_meta = f"{output_meta_path}/signal-only-kinfit.json"


In [4]:
# ROOT.EnableImplicitMT(n_threads)
environ["OMP_NUM_THREADS"] = "6"

In [5]:
dataset = Dataset.from_json(dataset_path)

In [6]:
analysis = ReweightingHelper(dataset)

OBJ: TStyle	ildStyle	ILD Style : 0 at: 0xb7272a0


In [7]:
alt_setup_handler = AltSetupHandler("""
{
  "SM": {
    "mW": 80.419,
    "g1z": 1.0,
    "ka": 1.0,
    "la": 0.0
  },
"variations": [
    1e-08
  ]
}
""", mirror=False, combinations=False)
alt_configs = alt_setup_handler.get_alt_setup()
print(alt_configs)
analysis.initialise_omega_wrappers(alt_configs)

{'mW_pos_1em08': {'mW': 80.41900000999999, 'g1z': 1.0, 'ka': 1.0, 'la': 0.0}, 'g1z_pos_1em08': {'mW': 80.419, 'g1z': 1.00000001, 'ka': 1.0, 'la': 0.0}, 'ka_pos_1em08': {'mW': 80.419, 'g1z': 1.0, 'ka': 1.00000001, 'la': 0.0}, 'la_pos_1em08': {'mW': 80.419, 'g1z': 1.0, 'ka': 1.0, 'la': 1e-08}}


In [8]:
analysis.init_categories()
# check if we missed any processes
print(analysis.is_complete_categorisation())
signal_category = ["4f_sw_sl_signal"]

True


In [9]:
min_setups = ["nominal", "g1z_pos_1em08", "ka_pos_1em08", "la_pos_1em08"] + ["mW_pos_1em08"]

In [10]:
# define nominal beam lvecs
analysis.Define("nominal_beam_e_lvec", "ROOT::Math::PxPyPzMVector(+8.750143e-01, 0., +1.250000e+02, +5.109968e-04)")
analysis.Define("nominal_beam_p_lvec", "ROOT::Math::PxPyPzMVector(+8.750143e-01, 0., -1.250000e+02, +5.109968e-04)")

In [11]:
# reco configs
# clean jets
analysis.book_sqme(
                      [
                          "nominal_beam_e_lvec",
                          "nominal_beam_p_lvec",
                          "postfit_iso_lep_lvec",
                          "postfit_nu_lvec",
                          "postfit_R2Jet1_lvec",
                          "postfit_R2Jet2_lvec",
                      ],
                      "iso_lep_charge",
                      "kinfit_clean_reco",
                      alt_setups=min_setups,
                    #   categories=signal_category,
                      hels=True
                     )
analysis.book_sqme(
                      [
                          "nominal_beam_e_lvec",
                          "nominal_beam_p_lvec",
                          "postfit_iso_lep_lvec",
                          "postfit_nu_lvec",
                          "postfit_R2Jet2_lvec",
                          "postfit_R2Jet1_lvec",
                      ],
                      "iso_lep_charge",
                      "wj_kinfit_clean_reco",
                      alt_setups=min_setups,
                    #   categories=signal_category,
                      hels=True
                     )

In [12]:
if write_outputs:
    analysis.book_snapshots("events", output_path, output_meta, output_collections, no_rvec=no_rvec)

Info in <[ROOT.RDF] Info /tmp/root/spack-stage/spack-stage-root-6.38.00-2jf5cmbvudyjye7uzxzvsgmmlw2msfso/spack-build-2jf5cmb/include/ROOT/RDF/RInterface.hxx:1363 in auto ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager, void>::Snapshot(std::string_view, std::string_view, const ColumnNames_t &, const RSnapshotOptions &)::(anonymous class)::operator()() const [Proxied = ROOT::Detail::RDF::RLoopManager, DataSource = void]>: 
	In ROOT 6.38, the default compression settings of Snapshot have been changed from 101 (ZLIB with compression level 1, the TTree default) to 505 (ZSTD with compression level 5). This change may result in smaller Snapshot output dataset size by default. In order to suppress this message, set 'ROOT_RDF_SNAPSHOT_INFO=0' in your environment or set 'ROOT.RDF.Snapshot.Info: 0' in your .rootrc file.


In [13]:
%%time
analysis.run()

CPU times: user 2h 34min 17s, sys: 2min 1s, total: 2h 36min 18s
Wall time: 27min 49s


In [14]:
# if write_outputs:
    # analysis.check_snapshots("events", output_path, checked_output_meta)